<a href="https://colab.research.google.com/github/mdSagorhosen/RAG-Assisted-Phishing-Link-Analyzer-with-Heuristic-Risk-Scoring/blob/main/final_thesis_1097.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1

In [ ]:
!pip install -q huggingface_hub==0.23.4 sentence-transformers==2.7.0 gradio==4.44.0 faiss-cpu PyPDF2 pydantic==2.10.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Cell 2

In [ ]:
import re
import faiss
import numpy as np
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
import gradio as gr

embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ সব লোড হয়েছে, সমস্যা নেই")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ সব লোড হয়েছে, সমস্যা নেই


Cell 3

In [ ]:
default_knowledge = [
"Phishing links often use IP addresses instead of domain names.",
"Phishing URLs may contain '@' symbol to hide the real destination.",
"Legitimate websites usually use HTTPS, but phishing sites can fake HTTPS too, so HTTPS alone is not proof of safety.",
"Phishing domains often misspell popular brand names like paypa1.com or faceb00k.com.",
"URLs with too many subdomains or hyphens are suspicious.",
"Shortened URLs like bit.ly or tinyurl are commonly used to hide phishing destinations.",
"Phishing pages often urgently ask for login credentials, OTP, or bank details.",
"Suspicious URLs may use unusual top-level domains like .xyz, .top, .club used heavily in scams.",
"Long random-looking URLs with numbers and letters mixed are often auto-generated phishing links.",
"Official bank or government sites never ask for password via email link."
]

embeddings = embedder.encode(default_knowledge)
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings).astype('float32'))

print("✅ Vector DB তৈরি হয়েছে, total entries:", len(default_knowledge))

✅ Vector DB তৈরি হয়েছে, total entries: 10


Cell 4

In [ ]:
def url_heuristics(url):
    flags = []

    if '@' in url:
        flags.append(("URL-এ '@' সিম্বল আছে — আসল লিংক আড়াল করার কমন কৌশল।", 3))

    if re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', url):
        flags.append(("ডোমেইনের বদলে সরাসরি IP address ব্যবহার হয়েছে।", 3))

    if url.count('-') > 5:
        flags.append(("URL-এ অনেক বেশি হাইফেন (-) আছে।", 1))

    if any(s in url.lower() for s in ['bit.ly', 'tinyurl', 'goo.gl', 't.co']):
        flags.append(("এটা shortened URL — আসল destination আড়াল করা থাকতে পারে।", 2))

    if re.search(r'\.(xyz|top|club|tk|click|info)(\/|$)', url.lower()):
        flags.append(("সন্দেহজনক top-level domain (.xyz/.top/.club ইত্যাদি)।", 3))

    if len(url) > 120:
        flags.append(("URL টা অস্বাভাবিক লম্বা — auto-generated হতে পারে।", 1))

    if not url.lower().startswith('https://'):
        flags.append(("URL-টা HTTPS দিয়ে শুরু হয়নি — নিরাপত্তা ঝুঁকি থাকতে পারে।", 1))

    brands = ['paypal', 'facebook', 'google', 'microsoft', 'amazon', 'apple', 'netflix']
    domain_match = re.search(r'https?://([^/]+)', url.lower())
    domain = domain_match.group(1) if domain_match else url.lower()
    domain_parts = domain.split('.')
    main_domain = ".".join(domain_parts[-2:]) if len(domain_parts) >= 2 else domain

    for b in brands:
        if b in domain and main_domain != f"{b}.com":
            flags.append((f"'{b}' নাম ডোমেইনে আছে কিন্তু আসল ডোমেইন '{b}.com' নয় — brand spoofing হতে পারে।", 5))

    return flags

def retrieve_context(url, k=3):
    q_embed = embedder.encode([url])
    distances, indices = index.search(np.array(q_embed).astype('float32'), k)
    return [default_knowledge[i] for i in indices[0]]

print("✅ Heuristics ready")

✅ Heuristics ready


Cell 5

In [ ]:
def analyze_link(url):
    if not url or not url.strip():
        return "একটা লিংক দিন।"

    url = url.strip()
    flags = url_heuristics(url)
    context = retrieve_context(url)

    risk_score = sum(w for _, w in flags)

    if risk_score == 0:
        verdict = "✅ নিরাপদ মনে হচ্ছে"
    elif risk_score <= 2:
        verdict = "⚠️ হালকা সন্দেহজনক — তবে সাধারণত ঝুঁকিপূর্ণ না"
    elif risk_score <= 5:
        verdict = "🟠 মাঝারি সন্দেহজনক — সাবধানে থাকুন"
    else:
        verdict = "🚨 উচ্চ সম্ভাবনা — এটা ফিশিং লিংক হতে পারে"

    flags_text = "\n".join(f"- {f} (risk +{w})" for f, w in flags) if flags else "- কোনো সরাসরি সন্দেহজনক নিশান পাওয়া যায়নি।"
    context_text = "\n".join(f"- {c}" for c in context)

    report = f"""🔗 URL: {url}

📊 সিদ্ধান্ত: {verdict}
🔢 Risk Score: {risk_score}

🚩 ধরা পড়া সিগন্যাল:
{flags_text}

📚 সংশ্লিষ্ট নিয়ম (RAG থেকে পাওয়া):
{context_text}

⚠️ এটা ১০০% নিশ্চিত গ্যারান্টি না। সন্দেহ হলে লিংকে ক্লিক করবেন না।
"""
    return report

def analyze_pdf(file):
    try:
        # Gradio ৪-এ ফাইল অবজেক্টটি সরাসরি ফাইলের পাথ (string) বা অন্য কোনো ফরম্যাটে আসতে পারে।
        # সেটিকে নিরাপদে হ্যান্ডেল করার জন্য ফাইলপাথ রিট্রিভ করার নিচের লজিকটি ব্যবহার করা হলো:
        if isinstance(file, str):
            filepath = file
        elif hasattr(file, 'name'):
            filepath = file.name
        elif hasattr(file, 'path'):
            filepath = file.path
        else:
            filepath = str(file)

        reader = PdfReader(filepath)
        text = ""
        for page in reader.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    except Exception as e:
        return f"PDF পড়তে সমস্যা হয়েছে: {e}"

    urls = re.findall(r'https?://[^\s)>\]"\'<]+', text)
    if not urls:
        return "এই PDF-এ কোনো লিংক পাওয়া যায়নি।"

    urls = list(dict.fromkeys(urls))[:10]
    report = f"📄 PDF-এ {len(urls)} টা ইউনিক লিংক পাওয়া গেছে (প্রথম ১০টা চেক করা হলো):\n\n"
    for u in urls:
        report += analyze_link(u) + "\n" + ("-"*40) + "\n"
    return report

print("✅ Analyze functions ready")

✅ Analyze functions ready


Cell 6

In [ ]:
with gr.Blocks(title="Phishing Link Detector") as demo:
    gr.Markdown("# 🛡️ Phishing Link Detector (RAG-based)")
    gr.Markdown("কোনো লিংক বা PDF দিন — সিস্টেম চেক করে বলে দেবে এটা ফিশিং নাকি নিরাপদ।")

    with gr.Tab("🔗 Link Check"):
        url_input = gr.Textbox(label="লিংক দিন", placeholder="https://example.com")
        url_btn = gr.Button("Analyze")
        url_output = gr.Textbox(label="রিপোর্ট", lines=15)
        url_btn.click(analyze_link, inputs=url_input, outputs=url_output)

    with gr.Tab("📄 PDF Check"):
        pdf_input = gr.File(label="PDF আপলোড করুন")
        pdf_btn = gr.Button("Analyze PDF")
        pdf_output = gr.Textbox(label="রিপোর্ট", lines=20)
        pdf_btn.click(analyze_pdf, inputs=pdf_input, outputs=pdf_output)

# show_api=False দিয়ে Pydantic স্কিমা সংক্রান্ত বাগটি এড়ানো হয়েছে এবং share=True দিয়ে কোলাবের জন্য টানেলিং নিশ্চিত করা হয়েছে।
demo.launch(share=True, show_api=False)

/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(
ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 90, in __call__
    await self.middleware_stack(scope, r

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://d396a0cbf5594acbf7.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Streaming output truncated to the last 5000 lines.
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/starlette/templating.py", line 148, in TemplateResponse
    template = self.get_template(name)
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/starlette/templating.py", line 115, in get_template
    return self.env.get_template(name)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/jinja2/environment.py", line 1016, in get_template
    return self._load_template(name, globals)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/jinja2/environment.py", line 964, in _load_template
    template = self.cache.get(cache_key)
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/jinja2/utils.py", line 477, in get
    return self[key]
           ~~~~^^^^^
  File "/usr/local/lib/python3.12/dist-packages/jinj